In [ ]:
import torch
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from torchvision import models
import os

# Check for GPU availability to ensure faster training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# ==========================================
# 1. Paths & Transform (Added Augmentation)
# ==========================================
# %% [markdown]
# ## 1. Paths & Transform (Added Augmentation)
# Defining data paths and necessary transformations. 
# Training transforms include augmentation to reduce overfitting and improve generalization.

train_path = r"D:\Deep Learning\ball_classification_dataset\train"
test_path = r"D:\Deep Learning\ball_classification_dataset\test"

image_size = 224 

train_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomResizedCrop(image_size, scale=(0.8, 1.0)), # Random crop
    transforms.ColorJitter(brightness=0.2, contrast=0.2), # Random brightness/contrast
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
# ==========================================
# 2. Load Dataset
# ==========================================
train_dataset = ImageFolder(root=train_path, transform=train_transform)
test_dataset = ImageFolder(root=test_path, transform=test_transform)

num_classes = len(train_dataset.classes)
class_names = train_dataset.classes
print(f"Total classes found: {num_classes}")
print(f"Classes: {class_names}")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [ ]:
# ==========================================
# 3. Model Setup
# ==========================================

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_ftrs = model.fc.in_features

# Modifying the final layer with Dropout
model.fc = torch.nn.Sequential(
    torch.nn.Dropout(0.7),
    torch.nn.Linear(num_ftrs, num_classes)
)

model = model.to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.00001, weight_decay=1e-3)

In [ ]:
# ==========================================
# 4. Training Loop
# ==========================================
epochs = 50
patience = 5  
counter = 0
best_test_loss = float('inf')

history = {'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': []}

print("Starting Training...")
for epoch in range(epochs):
    # --- PHASE: TRAINING ---
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        train_correct += (preds == y).sum().item()
        train_total += y.size(0)

    # --- PHASE: VALIDATION (TEST) ---
    model.eval()
    test_loss, test_correct, test_total = 0, 0, 0
    
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            loss = criterion(outputs, y)
            
            test_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            test_correct += (preds == y).sum().item()
            test_total += y.size(0)

    # Calculate averages
    epoch_train_loss = train_loss / len(train_loader)
    epoch_test_loss = test_loss / len(test_loader)
    epoch_train_acc = train_correct / train_total
    epoch_test_acc = test_correct / test_total

    history['train_loss'].append(epoch_train_loss)
    history['test_loss'].append(epoch_test_loss)
    history['train_acc'].append(epoch_train_acc)
    history['test_acc'].append(epoch_test_acc)

    print(f"Epoch {epoch+1}/{epochs} | Train Acc: {epoch_train_acc*100:.2f}% | Test Acc: {epoch_test_acc*100:.2f}%")

    # Early Stopping Logic
    if epoch_test_loss < best_test_loss:
        best_test_loss = epoch_test_loss
        torch.save(model.state_dict(), 'best_ball_model.pth') 
        counter = 0
        print(f"--> Test Loss improved. Best model saved!")
    else:
        counter += 1
        print(f"--> No improvement for {counter} consecutive epochs.")
        if counter >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

In [ ]:
# ==========================================
# 5. Plotting Results
# ==========================================
plt.figure(figsize=(12, 5))

# Loss Plot
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['test_loss'], label='Test Loss')
plt.title('Loss Analysis')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Accuracy Plot
plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Train Acc')
plt.plot(history['test_acc'], label='Test Acc')
plt.title('Accuracy Analysis')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.show()

In [ ]:
# ==========================================
# 6. Detection Results Visualization
# ==========================================
model.load_state_dict(torch.load('best_ball_model.pth'))
model.eval()

images_per_class = 4 
num_columns = 5       
classes_to_show = class_names[:num_columns] 

plt.figure(figsize=(15, 12))

with torch.no_grad():
    for col_idx, target_class in enumerate(classes_to_show):
        class_idx = class_names.index(target_class)
        found_images = 0
        
        for i in range(len(test_dataset)):
            img, label = test_dataset[i]
            if label == class_idx:
                img_input = img.unsqueeze(0).to(device)
                output = model(img_input)
                _, pred = torch.max(output, 1)
                
                # Denormalize for correct display
                img_show = img.permute(1, 2, 0).numpy()
                img_show = img_show * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]
                img_show = img_show.clip(0, 1)

                plt.subplot(images_per_class, num_columns, found_images * num_columns + col_idx + 1)
                plt.imshow(img_show)
                color = 'green' if pred.item() == label else 'red'
                
                plt.title(f"T: {target_class}\nP: {class_names[pred.item()]}", color=color, fontsize=9)
                plt.axis('off')
                found_images += 1
            
            if found_images == images_per_class:
                break

plt.tight_layout()
plt.suptitle(f"Detection Results for Best Model", fontsize=16, y=1.02)
plt.show()